In [1]:
# =========================================================
# CELL 1: SETUP
# =========================================================
import os, glob, random, time
from PIL import Image

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.optim import Adam, AdamW
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io
from torchvision.models import maxvit_t, MaxVit_T_Weights

# Seed
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 55)
print(f"PyTorch        : {torch.__version__}")
print(f"Torchvision    : {torchvision.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU[{i}]       : {p.name} ({p.total_memory/1024**3:.1f} GB)")
print(f"→ device = {device}")
print("=" * 55)

PyTorch        : 2.11.0+cu128
Torchvision    : 0.26.0+cu128
CUDA available : True
GPU count      : 2
  GPU[0]       : Tesla T4 (14.6 GB)
  GPU[1]       : Tesla T4 (14.6 GB)
→ device = cuda


In [2]:
# =========================================================
# CELL 2: CONFIG + UTILS 
# =========================================================
import os
import torch
import torch.nn as nn
from tqdm.auto import tqdm

CFG = {
    "data_root":    "/kaggle/input/datasets/moltean/fruits/fruits-360_100x100/fruits-360",
    "train_dir":    "Training",
    "valid_dir":    "Test",
    "img_size":     224,
    "batch_size":   64,          # Đã hạ xuống 64 để chống OOM khi Fine-tune
    "num_workers":  4,           
    "epochs":       5,
    "lr_head":      1e-3,
    "lr_finetune":  1e-4,
    "weight_decay": 1e-4,
    "out_dir":      "/kaggle/working",
}
DATA_ROOT = CFG["data_root"]
print("Data root exists:", os.path.exists(DATA_ROOT))

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ================== UTILS ==================
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N


def train(model, train_loader, train_N, random_trans, optimizer, loss_function, device):
    loss_sum = 0.0
    accuracy = 0.0
    model.train()

    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=device).view(1, 3, 1, 1)
    
    scaler = torch.amp.GradScaler("cuda")

    pbar = tqdm(train_loader, desc="Train", leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True).float()
        y = y.to(device, non_blocking=True)

        x = random_trans(x)            
        x = (x - mean) / std           

        optimizer.zero_grad(set_to_none=True) 
        
        with torch.amp.autocast("cuda"):
            output = model(x)
            batch_loss = loss_function(output, y)

        scaler.scale(batch_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
        pbar.set_postfix(loss=f"{batch_loss.item():.4f}")

    avg_loss = loss_sum / len(train_loader)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(avg_loss, accuracy))
    return avg_loss, accuracy


def validate(model, valid_loader, valid_N, loss_function, device, val_trans=None):
    loss_sum = 0.0
    accuracy = 0.0
    model.eval()

    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=device).view(1, 3, 1, 1)

    with torch.no_grad():
        pbar = tqdm(valid_loader, desc="Valid", leave=False)
        for x, y in pbar:
            x = x.to(device, non_blocking=True).float()
            y = y.to(device, non_blocking=True)

            if val_trans is not None:
                x = val_trans(x)       
            x = (x - mean) / std       

            with torch.amp.autocast("cuda"):
                output = model(x)

            # FIX LỖI INF LOSS: Ép output về float32 trước khi tính loss
            loss_sum += loss_function(output.float(), y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)

    avg_loss = loss_sum / len(valid_loader)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(avg_loss, accuracy))
    return avg_loss, accuracy

print("CFG + utils ready (Bản hoàn hảo - Không OOM, Không Inf) ✅")

Data root exists: True
CFG + utils ready (Bản hoàn hảo - Không OOM, Không Inf) ✅


In [3]:
# =========================================================
# CELL 3: LOAD MAXVIT PRETRAINED
# =========================================================
weights = MaxVit_T_Weights.IMAGENET1K_V1
maxvit_model = maxvit_t(weights=weights).to(device)
print("MaxViT-T loaded | classifier:", maxvit_model.classifier)

Downloading: "https://download.pytorch.org/models/maxvit_t-bc5ab103.pth" to /root/.cache/torch/hub/checkpoints/maxvit_t-bc5ab103.pth


100%|██████████| 119M/119M [00:00<00:00, 224MB/s]


MaxViT-T loaded | classifier: Sequential(
  (0): AdaptiveAvgPool2d(output_size=1)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): Tanh()
  (5): Linear(in_features=512, out_features=1000, bias=False)
)


In [4]:
# =========================================================
# CELL 4: FREEZE BACKBONE
# =========================================================
maxvit_model.requires_grad_(False)
print("Backbone requires_grad:", next(iter(maxvit_model.parameters())).requires_grad)

Backbone requires_grad: False


In [5]:
# =========================================================
# CELL 5: REPLACE CLASSIFIER HEAD
# =========================================================
DATA_LABELS = sorted(os.listdir(os.path.join(DATA_ROOT, "Training")))
N_CLASSES = len(DATA_LABELS)
print(f"N_CLASSES = {N_CLASSES}")

in_features = maxvit_model.classifier[-1].in_features
maxvit_model.classifier[-1] = nn.Linear(in_features, N_CLASSES)

my_model = maxvit_model.to(device)
print("New head:", my_model.classifier[-1])

N_CLASSES = 278
New head: Linear(in_features=512, out_features=278, bias=True)


In [6]:
# =========================================================
# ĐOẠN CODE XEM THÔNG SỐ MÔ HÌNH MAXVIT
# =========================================================
def print_model_summary(model):
    print("=" * 60)
    print("THÔNG SỐ MÔ HÌNH MAXVIT")
    print("=" * 60)
    
    # 1. Đếm số lượng tham số
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    
    print(f"📌 Tổng số tham số     : {total_params:,}")
    print(f"🔥 Đang train (Train)  : {trainable_params:,}")
    print(f"❄️  Đang đóng băng      : {frozen_params:,}")
    print("-" * 60)
    
    # 2. In lớp đầu tiên (Stem)
    print("🔹 LỚP ĐẦU TIÊN (Stem/Patch Embedding):")
    try:
        # MaxViT trong torchvision thường lưu lớp Stem ở features[0]
        print(model.features[0])
    except Exception as e:
        print(f"  (Lỗi không đọc được lớp đầu: {e})")
    print("-" * 60)
    
    # 3. In lớp cuối cùng (Classifier)
    print("🔸 LỚP CUỐI CÙNG (Classifier Head):")
    print(model.classifier)
    print("=" * 60)

# Gọi hàm để in:
print_model_summary(my_model)

THÔNG SỐ MÔ HÌNH MAXVIT
📌 Tổng số tham số     : 30,550,238
🔥 Đang train (Train)  : 142,614
❄️  Đang đóng băng      : 30,407,624
------------------------------------------------------------
🔹 LỚP ĐẦU TIÊN (Stem/Patch Embedding):
  (Lỗi không đọc được lớp đầu: 'MaxVit' object has no attribute 'features')
------------------------------------------------------------
🔸 LỚP CUỐI CÙNG (Classifier Head):
Sequential(
  (0): AdaptiveAvgPool2d(output_size=1)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): Tanh()
  (5): Linear(in_features=512, out_features=278, bias=True)
)


In [7]:
import torch
!pip install --upgrade onnx onnxscript -q
# Đảm bảo model ở chế độ eval
my_model.eval()

# Tạo một tensor giả (dummy input) giống với ảnh thật (Batch=1, 3 kênh màu, 224x224)
dummy_input = torch.randn(1, 3, 224, 224).to(device)

onnx_path = "/kaggle/working/maxvit_fruit360.onnx"

# Xuất mô hình
torch.onnx.export(
    my_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"✅ Đã xuất mô hình thành công: {onnx_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.2/192.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-cloud-aiplatform 2.3.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but y

/tmp/ipykernel_23/1481951643.py:12: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1030 09:54:02.857000 23 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MaxVit([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MaxVit([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str,

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Đã xuất mô hình thành công: /kaggle/working/maxvit_fruit360.onnx


In [8]:
# =========================================================
# CELL 6: LOSS + OPTIMIZER 
# =========================================================
loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = Adam(
    filter(lambda p: p.requires_grad, my_model.parameters()),
    lr=CFG["lr_head"],
    weight_decay=CFG["weight_decay"],
)

# ĐÃ TẮT TORCH.COMPILE ĐỂ TRÁNH LỖI RESHAPE CỦA MAXVIT
# my_model = torch.compile(my_model) 
print("Đã bỏ qua torch.compile, chạy chế độ Eager bình thường.")

Đã bỏ qua torch.compile, chạy chế độ Eager bình thường.


In [9]:
# =========================================================
# CELL 7: TRANSFORMS 
# =========================================================
random_trans = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    # KHÔNG dùng ColorJitter — color là discriminator của Fruit-360
])

val_trans = transforms.Compose([
    transforms.Resize((224, 224)),
])

In [10]:
# =========================================================
# CELL 8: DATASET (LAZY LOAD) + DATALOADER (ĐÃ TỐI ƯU)
# =========================================================
import glob
import time
from torch.utils.data import Dataset, DataLoader
import torchvision.io as tv_io


class MyDataset(Dataset):
    def __init__(self, data_dir, class_to_idx):
        self.samples = []
        skipped = 0
        for label in os.listdir(data_dir):
            if label not in class_to_idx:
                skipped += 1
                continue
            l_idx = class_to_idx[label]
            paths = glob.glob(os.path.join(data_dir, label, "*"), recursive=True)
            for p in paths:
                if p.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((p, l_idx))
        if skipped:
            print(f"⚠️  Bỏ qua {skipped} class không có trong Training")
        print(f"📁 {data_dir.split('/')[-1]:10s} | {len(self.samples):>7,} ảnh")

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = tv_io.read_image(path, tv_io.ImageReadMode.RGB)
        except Exception as e:
            # Nếu ảnh hỏng → trả về ảnh đen (không crash giữa lúc train)
            print(f"⚠️  Ảnh lỗi: {path} — dùng ảnh đen thay thế")
            img = torch.zeros(3, 224, 224, dtype=torch.uint8)
        return img, torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return len(self.samples)


# class_to_idx lấy từ DATA_LABELS (đã tạo ở Cell 5)
class_to_idx = {c: i for i, c in enumerate(DATA_LABELS)}
print(f"class_to_idx size: {len(class_to_idx)}")

n = CFG["batch_size"]
n_workers = CFG["num_workers"]

# ⚠️ FIX: persistent_workers chỉ hoạt động khi num_workers > 0
use_persistent = n_workers > 0

train_path = os.path.join(DATA_ROOT, CFG["train_dir"])
valid_path = os.path.join(DATA_ROOT, CFG["valid_dir"])

t0 = time.time()
train_data = MyDataset(train_path, class_to_idx)
valid_data = MyDataset(valid_path, class_to_idx)
print(f"⏱  Init dataset: {time.time()-t0:.1f}s")

# Train loader (shuffle=True, drop_last=True để batch size cố định)
train_loader = DataLoader(
    train_data,
    batch_size=n,
    shuffle=True,
    num_workers=n_workers,
    pin_memory=True,
    drop_last=True,
    persistent_workers=use_persistent,
    prefetch_factor=2 if n_workers > 0 else None,   # prefetch 2 batch trước
)

# Valid loader (shuffle=False, giữ nguyên số mẫu)
valid_loader = DataLoader(
    valid_data,
    batch_size=n,
    shuffle=False,
    num_workers=n_workers,
    pin_memory=True,
    persistent_workers=use_persistent,
    prefetch_factor=2 if n_workers > 0 else None,
)

train_N = len(train_loader.dataset)
valid_N = len(valid_loader.dataset)
print(f"\n✅ train_N={train_N:,} | valid_N={valid_N:,} | steps/epoch={len(train_loader)}")
print(f"   num_workers={n_workers} | batch={n} | persistent={use_persistent}")

class_to_idx size: 278
📁 Training   | 149,465 ảnh
⚠️  Bỏ qua 1 class không có trong Training
📁 Test       |  49,640 ảnh
⏱  Init dataset: 100.5s

✅ train_N=149,465 | valid_N=49,640 | steps/epoch=2335
   num_workers=4 | batch=64 | persistent=True


In [11]:
# =========================================================
# CELL 9A: TRAIN STAGE 1
# =========================================================
import time
import os
import torch

best_acc = 0.0
epochs = CFG["epochs"]
total_start = time.time()

# 1. Khởi tạo history để lưu kết quả
history = {"epoch": [], "tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}

for epoch in range(epochs):
    print(f"\n===== Epoch {epoch+1}/{epochs} =====")
    t0 = time.time()

    tr_loss, tr_acc = train(my_model, train_loader, train_N,
                            random_trans, optimizer, loss_function, device)
    va_loss, va_acc = validate(my_model, valid_loader, valid_N,
                               loss_function, device, val_trans=val_trans)

    elapsed = time.time() - t0
    remaining = elapsed * (epochs - epoch - 1)
    print(f"⏱  Epoch time: {elapsed:.1f}s | ETA: {remaining/60:.1f} phút")

    # 2. Ghi lại history
    history["epoch"].append(epoch + 1)
    history["tr_loss"].append(tr_loss)
    history["tr_acc"].append(tr_acc)
    history["va_loss"].append(va_loss)
    history["va_acc"].append(va_acc)

    if va_acc > best_acc:
        best_acc = va_acc
        # 3. Đồng bộ key lưu là "model_state"
        torch.save({"model_state": my_model.state_dict(),
                    "classes": DATA_LABELS,
                    "acc": best_acc},
                   os.path.join(CFG["out_dir"], "best_head.pt"))
        print(f"   ↳ saved (best={best_acc:.4f})")

print(f"\n✅ Tổng: {(time.time()-total_start)/60:.1f} phút")
print("Best Stage 1:", best_acc)


===== Epoch 1/5 =====


Train:   0%|          | 0/2335 [00:00<?, ?it/s]

Train - Loss: 1.7056 Accuracy: 0.8686


Valid:   0%|          | 0/776 [00:00<?, ?it/s]

Valid - Loss: 1.6775 Accuracy: 0.8311
⏱  Epoch time: 7201.5s | ETA: 480.1 phút
   ↳ saved (best=0.8311)

===== Epoch 2/5 =====


Train:   0%|          | 0/2335 [00:00<?, ?it/s]

... (11 outputs left out)

Train:   0%|          | 0/2335 [00:00<?, ?it/s]

Train - Loss: 1.1970 Accuracy: 0.9706


Valid:   0%|          | 0/776 [00:00<?, ?it/s]

Valid - Loss: 1.5427 Accuracy: 0.8798
⏱  Epoch time: 3097.7s | ETA: 0.0 phút

✅ Tổng: 320.7 phút
Best Stage 1: 0.887671232876715


In [12]:
# =========================================================
# CELL 9B: FINE-TUNE (BẢN FIX 100% OOM KHI RUN ALL)
# =========================================================
import time
import os
import torch
import gc
from torch.optim import Adam

print("=" * 60)
print("BẮT ĐẦU FINE-TUNE TOÀN BỘ MÔ HÌNH")
print("=" * 60)

# --- 0. KIỂM TRA AN TOÀN ---
if 'best_acc' not in globals():
    best_acc = 0.0
if 'history' not in globals() or len(history.get("epoch", [])) == 0:
    history = {"epoch": [], "tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}

# --- 1. DỌN DẸP BỘ NHỚ TRIỆT ĐỂ TRƯỚC KHI MỞ BĂNG ---
# XÓA OPTIMIZER CỦA STAGE 1 ĐỂ GIẢI PHÓNG VRAM
if 'optimizer' in globals():
    del optimizer
    print("🧹 Đã xóa Optimizer của Stage 1.")

gc.collect()
torch.cuda.empty_cache()
free_vram = torch.cuda.mem_get_info()[0] / (1024**3)
print(f"🧹 Đã dọn dẹp VRAM. Còn trống: {free_vram:.2f} GB")

if free_vram < 8.0:
    print("⚠️ CẢNH BÁO: VRAM còn quá ít (< 8GB). Nguy cơ cao bị OOM khi Fine-tune!")
    print("→ Khuyên bạn nên Stop Session, vào Cell 2 giảm 'batch_size' xuống 32 hoặc 64, sau đó chạy lại.")

# --- 2. MỞ BĂNG TOÀN BỘ ---
for p in my_model.parameters():
    p.requires_grad = True
print("✅ Đã mở băng toàn bộ Model (Backbone + Head)")

# --- 3. OPTIMIZER MỚI VÀ VÒNG LẶP TRAIN ---
optimizer_ft = Adam(my_model.parameters(), lr=CFG["lr_finetune"], weight_decay=CFG["weight_decay"])
best_acc_ft = best_acc
FT_EPOCHS = 2

total_start = time.time()

for epoch in range(FT_EPOCHS):
    print(f"\n===== Fine-tune Epoch {epoch+1}/{FT_EPOCHS} =====")
    t0 = time.time()

    try:
        tr_loss, tr_acc = train(my_model, train_loader, train_N,
                                random_trans, optimizer_ft, loss_function, device)
        va_loss, va_acc = validate(my_model, valid_loader, valid_N,
                                   loss_function, device, val_trans=val_trans)
    except RuntimeError as e:
        if "out of memory" in str(e):
            print("\n❌ LỖI: Hết bộ nhớ VRAM (OOM)!")
            print("→ Hãy Stop Session, giảm batch_size xuống 32 trong Cell 2, rồi chạy lại.")
            break
        else:
            raise e

    elapsed = time.time() - t0
    print(f"⏱  Epoch time: {elapsed:.1f}s")

    # Nối tiếp history
    next_epoch = history["epoch"][-1] + 1 if history["epoch"] else 1
    history["epoch"].append(next_epoch)
    history["tr_loss"].append(tr_loss)
    history["tr_acc"].append(tr_acc)
    history["va_loss"].append(va_loss)
    history["va_acc"].append(va_acc)

    # Lưu model tốt nhất
    if va_acc > best_acc_ft:
        best_acc_ft = va_acc
        torch.save({"model_state": my_model.state_dict(),
                    "classes": DATA_LABELS,
                    "acc": best_acc_ft},
                   os.path.join(CFG["out_dir"], "best_ft.pt"))
        print(f"   ↳ 🔥 saved best_ft.pt (best={best_acc_ft:.4f})")

    # Dọn dẹp sau mỗi epoch
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Tổng thời gian Fine-tune: {(time.time()-total_start)/60:.1f} phút")
print(f"🏆 Độ chính xác cao nhất đạt được: {best_acc_ft*100:.2f}%")

BẮT ĐẦU FINE-TUNE TOÀN BỘ MÔ HÌNH
🧹 Đã xóa Optimizer của Stage 1.
🧹 Đã dọn dẹp VRAM. Còn trống: 14.28 GB
✅ Đã mở băng toàn bộ Model (Backbone + Head)

===== Fine-tune Epoch 1/2 =====


Train:   0%|          | 0/2335 [00:00<?, ?it/s]

Train - Loss: 0.9757 Accuracy: 0.9976


Valid:   0%|          | 0/776 [00:00<?, ?it/s]

Valid - Loss: 0.9751 Accuracy: 0.9943
⏱  Epoch time: 3923.7s
   ↳ 🔥 saved best_ft.pt (best=0.9943)

===== Fine-tune Epoch 2/2 =====


Train:   0%|          | 0/2335 [00:00<?, ?it/s]

Train - Loss: 0.9400 Accuracy: 0.9978


Valid:   0%|          | 0/776 [00:00<?, ?it/s]

Valid - Loss: 0.9806 Accuracy: 0.9942
⏱  Epoch time: 3849.5s

✅ Tổng thời gian Fine-tune: 129.6 phút
🏆 Độ chính xác cao nhất đạt được: 99.43%


In [13]:
# =========================================================
# CELL 10: TỔNG KẾT MAXVIT — EVAL + PSNR + BÁO CÁO + ĐỒ THỊ 
# (BẢN FIX DÀNH RIÊNG CHO SAVE VERSION / RUN ALL)
# =========================================================
import os, time
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg') # QUAN TRỌNG: Chạy ngầm, không cần giao diện, không bị treo
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report, confusion_matrix, top_k_accuracy_score
)
from tqdm.auto import tqdm
import torch

OUT_DIR = "/kaggle/working"
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'

# ==========================================
# PHẦN 1: IN KIẾN TRÚC LỚP ĐẦU & LỚP CUỐI
# ==========================================
print("=" * 60)
print("KIẾN TRÚC MAXVIT (LỚP ĐẦU & LỚP CUỐI)")
print("=" * 60)
print("\n--- LỚP ĐẦU TIÊN (Stem / Patch Embedding) ---")
if hasattr(my_model, 'stem'):
    print(my_model.stem)
elif hasattr(my_model, 'features'):
    print(my_model.features[0])
else:
    print(list(my_model.children())[0])

print("\n--- LỚP CUỐI CÙNG (Classifier Head / Filter) ---")
print(my_model.classifier)

# ==========================================
# PHẦN 2: KIỂM TRA VÀ TẠO HISTORY AN TOÀN
# ==========================================
if 'history' not in globals() or len(history.get("epoch", [])) == 0:
    print("⚠️ Không tìm thấy history, tạo mới để tránh lỗi vẽ đồ thị.")
    history = {"epoch": [], "tr_loss": [], "tr_acc": [], "va_loss": [], "va_acc": []}

# ==========================================
# PHẦN 3: LOAD BEST MODEL
# ==========================================
print("\n" + "=" * 60)
print("BƯỚC 1: LOAD BEST MODEL")
print("=" * 60)

CKPT = os.path.join(OUT_DIR, "best_ft.pt")
if not os.path.exists(CKPT):
    CKPT = os.path.join(OUT_DIR, "best_head.pt")
    print(f"⚠️ Không có best_ft.pt, dùng: {CKPT}")

ckpt = torch.load(CKPT, map_location=device)
my_model.load_state_dict(ckpt["model_state"])
my_model.eval()
saved_acc = ckpt.get("acc", 0.0)
print(f"Loaded   : {CKPT}")
print(f"Saved acc: {saved_acc*100:.2f}%")

# ==========================================
# PHẦN 4: HÀM TÍNH PSNR
# ==========================================
def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0: return float('inf')
    return (10 * torch.log10(1.0 / mse)).item()

# ==========================================
# PHẦN 5: INFERENCE TRÊN TEST SET + TÍNH PSNR
# ==========================================
print("\n" + "=" * 60)
print("BƯỚC 2: INFERENCE TRÊN TẬP TEST")
print("=" * 60)

preds, labels, probs = [], [], []
loss_sum = 0.0
valid_loss_count = 0
top5_correct = 0
total = 0
psnr_values = []

mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
std  = torch.tensor(IMAGENET_STD,  device=device).view(1, 3, 1, 1)

t0 = time.time()
with torch.no_grad():
    for x, y in tqdm(valid_loader, desc="Eval"):
        x = x.to(device).float()
        y = y.to(device)

        # 1. Tính PSNR chuẩn (trên ảnh gốc [0, 1] trước khi normalize)
        x_raw = x / 255.0
        x_noisy = torch.clamp(x_raw + torch.randn_like(x_raw) * 0.1, 0, 1)
        psnr_values.append(calculate_psnr(x_raw, x_noisy))

        # 2. Transform + normalize để đưa vào model
        x = val_trans(x)
        x = (x - mean) / std

        with torch.amp.autocast("cuda"):
            out = my_model(x)

        # 3. Tính loss (bỏ qua nếu loss bị inf hoặc nan)
        batch_loss = loss_function(out, y).item()
        if not (np.isinf(batch_loss) or np.isnan(batch_loss)):
            loss_sum += batch_loss
            valid_loss_count += 1

        p = torch.softmax(out, dim=1)
        probs.extend(p.cpu().numpy())
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())

        _, top5 = out.topk(5, dim=1)
        top5_correct += top5.eq(y.view(-1, 1)).sum().item()
        total += y.size(0)

preds  = np.array(preds)
labels = np.array(labels)
probs  = np.array(probs)

avg_loss = loss_sum / valid_loss_count if valid_loss_count > 0 else 0.0
top1_acc = (preds == labels).mean()
top5_acc = top5_correct / total
avg_psnr = float(np.mean(psnr_values))
print(f"Xong trong {time.time()-t0:.1f}s")
print(f"Avg PSNR: {avg_psnr:.2f} dB")

# ==========================================
# PHẦN 6: BẢNG TỔNG HỢP
# ==========================================
print("\n" + "=" * 60)
print("BƯỚC 3: BẢNG TỔNG HỢP")
print("=" * 60)

summary = pd.DataFrame({
    "Metric": ["Test Loss", "Top-1 Accuracy", "Top-5 Accuracy",
               "Avg PSNR (dB)", "Num samples", "Num classes"],
    "Value" : [f"{avg_loss:.4f}", f"{top1_acc*100:.2f}%", f"{top5_acc*100:.2f}%",
               f"{avg_psnr:.2f}", f"{len(labels):,}", f"{N_CLASSES}"],
})
print(summary.to_string(index=False))
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False)

# ==========================================
# PHẦN 7: CLASSIFICATION REPORT CHI TIẾT
# ==========================================
print("\n" + "=" * 60)
print(f"CLASSIFICATION REPORT CHI TIẾT ({N_CLASSES} lớp) — MAXVIT")
print("=" * 60)

report_dict = classification_report(
    labels, preds, labels=range(N_CLASSES), target_names=DATA_LABELS, digits=4, zero_division=0, output_dict=True
)
report_str = classification_report(
    labels, preds, labels=range(N_CLASSES), target_names=DATA_LABELS, digits=4, zero_division=0
)
report_df = pd.DataFrame(report_dict).transpose()

print(report_str[:3000])
print("\n... (xem file full bên dưới)")

with open(os.path.join(OUT_DIR, "maxvit_classification_report_full.txt"), "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("CLASSIFICATION REPORT — MAXVIT (PRETRAINED) TRÊN FRUIT-360\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Test Accuracy       : {top1_acc*100:.2f}%\n")
    f.write(f"Top-5 Accuracy      : {top5_acc*100:.2f}%\n")
    f.write(f"Precision (weighted): {report_dict['weighted avg']['precision']*100:.2f}%\n")
    f.write(f"Recall (weighted)   : {report_dict['weighted avg']['recall']*100:.2f}%\n")
    f.write(f"F1-Score (weighted) : {report_dict['weighted avg']['f1-score']*100:.2f}%\n")
    f.write(f"Avg PSNR (dB)       : {avg_psnr:.2f}\n")
    f.write(f"Num classes         : {N_CLASSES}\n")
    f.write(f"Num test samples    : {len(labels):,}\n\n")
    f.write("=" * 70 + "\nCHI TIẾT TỪNG LỚP\n" + "=" * 70 + "\n\n")
    f.write(report_str)

report_df.to_csv(os.path.join(OUT_DIR, "maxvit_classification_report.csv"))
print("📄 Đã lưu: maxvit_classification_report_full.txt + maxvit_classification_report.csv")

class_df = report_df.loc[DATA_LABELS, ["precision", "recall", "f1-score", "support"]].copy()
for col in ["precision", "recall", "f1-score", "support"]:
    class_df[col] = class_df[col].astype(float)
class_df = class_df.sort_values("f1-score", ascending=False)

# ==========================================
# PHẦN 8: CONFUSION MATRIX + TOP ERRORS
# ==========================================
cm = confusion_matrix(labels, preds, labels=range(N_CLASSES))
errors = []
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if i != j and cm[i, j] > 0:
            errors.append((DATA_LABELS[i], DATA_LABELS[j], int(cm[i, j])))

errors_df = pd.DataFrame(errors, columns=["True", "Predicted", "Count"]).sort_values("Count", ascending=False).head(15)
if len(errors_df) > 0:
    errors_df.to_csv(os.path.join(OUT_DIR, "maxvit_top_errors.csv"), index=False)

# ==========================================
# PHẦN 9: ĐỒ THỊ 1 — TRAINING CURVES
# ==========================================
print("\n" + "=" * 60)
print("BƯỚC 4: VẼ ĐỒ THỊ")
print("=" * 60)
print("Đang vẽ Đồ thị 1: Training Curves...")

if len(history["epoch"]) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    epochs_range = history["epoch"]

    ax1.plot(epochs_range, history["tr_loss"], "o-", label="Train Loss", color='#e74c3c', linewidth=2)
    ax1.plot(epochs_range, history["va_loss"], "s-", label="Valid Loss", color='#3498db', linewidth=2)
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("Loss qua các Epoch", fontweight="bold")
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs_range, [a*100 for a in history["tr_acc"]], "o-", label="Train Acc", color='#e74c3c', linewidth=2)
    ax2.plot(epochs_range, [a*100 for a in history["va_acc"]], "s-", label="Valid Acc", color='#3498db', linewidth=2)
    ax2.axhline(y=top1_acc*100, color='green', linestyle=':', label=f"Final Test = {top1_acc*100:.2f}%")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("Accuracy qua các Epoch", fontweight='bold')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle("Training Curves — MaxViT Pretrained", fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "maxvit_plot_1_training_curves.png"), dpi=130)
    plt.close() # Đóng figure để giải phóng RAM

# ==========================================
# PHẦN 10: ĐỒ THỊ 2 — LOSS vs PSNR
# ==========================================
print("Đang vẽ Đồ thị 2: Loss vs PSNR...")
fig, ax1 = plt.subplots(figsize=(11, 6))
color = 'tab:red'
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("Loss", color=color, fontsize=12)
if len(history["epoch"]) > 0:
    ax1.plot(history["epoch"], history["tr_loss"], "o-", color=color, label="Train Loss", linewidth=2)
    ax1.plot(history["epoch"], history["va_loss"], "s--", color='darkred', label="Valid Loss", linewidth=2)
else:
    ax1.text(0.5, 0.5, "Không có history để vẽ Loss", transform=ax1.transAxes, ha='center')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel("PSNR (dB)", color=color, fontsize=12)
ax2.axhline(y=avg_psnr, color=color, linestyle='-', linewidth=2, label=f"PSNR (avg={avg_psnr:.2f} dB)")
ax2.tick_params(axis='y', labelcolor=color)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.title("Đồ thị Loss vs PSNR", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "maxvit_plot_2_loss_vs_psnr.png"), dpi=130)
plt.close()

# ==========================================
# PHẦN 11: ĐỒ THỊ 3 — PRECISION / RECALL / F1
# ==========================================
print("Đang vẽ Đồ thị 3: Precision/Recall/F1...")
fig, ax = plt.subplots(figsize=(14, 6))
np.random.seed(42)
sample_idx = np.random.choice(len(DATA_LABELS), min(40, len(DATA_LABELS)), replace=False)
sample_df = class_df.iloc[sample_idx].sort_values("f1-score")

x = np.arange(len(sample_df))
width = 0.27
ax.bar(x - width, sample_df["precision"], width, label="Precision", color='#3498db')
ax.bar(x,          sample_df["recall"],   width, label="Recall",   color='#e67e22')
ax.bar(x + width,  sample_df["f1-score"], width, label="F1-Score", color='#2ecc71')

ax.set_xlabel("Class"); ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1-Score (40 class mẫu — MaxViT)", fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(sample_df.index, rotation=90, fontsize=8)
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "maxvit_plot_3_precision_recall_f1.png"), dpi=130)
plt.close()

# ==========================================
# PHẦN 12: ĐỒ THỊ 4 — PHÂN BỐ F1-SCORE
# ==========================================
print("Đang vẽ Đồ thị 4: Phân bố F1-Score...")
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(class_df["f1-score"], bins=40, color='#9b59b6', edgecolor='black', alpha=0.7)
ax.axvline(class_df["f1-score"].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {class_df["f1-score"].mean():.3f}')
ax.axvline(class_df["f1-score"].median(), color='orange', linestyle=':', linewidth=2, label=f'Median = {class_df["f1-score"].median():.3f}')
ax.set_xlabel("F1-Score"); ax.set_ylabel("Số lớp")
ax.set_title(f"Phân bố F1-Score qua {N_CLASSES} lớp — MaxViT", fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "maxvit_plot_4_f1_distribution.png"), dpi=130)
plt.close()

# ==========================================
# PHẦN 13: ĐỒ THỊ 5 — CONFUSION MATRIX
# ==========================================
print("Đang vẽ Đồ thị 5: Confusion Matrix...")
TOP_N = 30
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm[:TOP_N, :TOP_N], annot=False, cmap="Blues", cbar_kws={"label": "Số mẫu"},
            xticklabels=DATA_LABELS[:TOP_N], yticklabels=DATA_LABELS[:TOP_N], ax=ax, square=True)
ax.set_title(f"Confusion Matrix — {TOP_N} lớp đầu tiên (MaxViT)", fontweight='bold')
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.xticks(rotation=90, fontsize=8); plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "maxvit_plot_5_confusion_matrix.png"), dpi=130)
plt.close()

# ==========================================
# PHẦN 14: ĐỒ THỊ 6 — TOP 15 CẶP CLASS HAY NHẦM
# ==========================================
if len(errors_df) > 0:
    print("Đang vẽ Đồ thị 6: Top lỗi...")
    fig, ax = plt.subplots(figsize=(13, 7))
    errors_df["pair"] = errors_df["True"] + " → " + errors_df["Predicted"]
    sns.barplot(x="Count", y="pair", data=errors_df, palette="OrRd_r", ax=ax)
    ax.set_title("TOP 15 CẶP CLASS HAY NHẦM NHẤT (MaxViT)", fontweight='bold')
    ax.set_xlabel("Số lần nhầm"); ax.set_ylabel("Cặp (True → Predicted)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "maxvit_plot_6_top_errors.png"), dpi=130)
    plt.close()

# ==========================================
# PHẦN 15: ĐỒ THỊ 7 — TOP-K ACCURACY
# ==========================================
print("Đang vẽ Đồ thị 7: Top-K Accuracy...")
ks = [1, 3, 5, 10, 20]
top_k_accs = []
for k in ks:
    acc = top_k_accuracy_score(labels, probs, k=k, labels=range(N_CLASSES))
    top_k_accs.append(acc * 100)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#3498db', '#2980b9', '#1abc9c', '#16a085', '#27ae60']
bars = ax.bar([f"Top-{k}" for k in ks], top_k_accs, color=colors)
for bar, acc in zip(bars, top_k_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc:.2f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_xlabel("Top-K"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Top-K Accuracy (MaxViT)", fontweight='bold')
ax.set_ylim([0, 105]); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "maxvit_plot_7_topk_accuracy.png"), dpi=130)
plt.close()

# ==========================================
# PHẦN 16: SAVE MODEL FINAL
# ==========================================
final_path = os.path.join(OUT_DIR, "maxvit_fruit360_final.pt")
torch.save({
    "model_state": my_model.state_dict(),
    "classes": DATA_LABELS,
    "val_acc": saved_acc,
    "top1_acc": float(top1_acc),
    "top5_acc": float(top5_acc),
    "avg_psnr": float(avg_psnr),
}, final_path)
print(f"\n✅ Đã lưu Model Final → {final_path}")

# ==========================================
# PHẦN 17: CHECKLIST FILE OUTPUT
# ==========================================
print("\n" + "=" * 60)
print("CHECKLIST FILE TRONG /kaggle/working/")
print("=" * 60)

required = [
    "maxvit_fruit360_final.pt",
    "summary.csv",
    "maxvit_classification_report_full.txt",
    "maxvit_classification_report.csv",
    "maxvit_top_errors.csv",
    "maxvit_plot_1_training_curves.png",
    "maxvit_plot_2_loss_vs_psnr.png",
    "maxvit_plot_3_precision_recall_f1.png",
    "maxvit_plot_4_f1_distribution.png",
    "maxvit_plot_5_confusion_matrix.png",
    "maxvit_plot_6_top_errors.png",
    "maxvit_plot_7_topk_accuracy.png",
]

missing = 0
for f in required:
    p = os.path.join(OUT_DIR, f)
    if os.path.exists(p):
        size = os.path.getsize(p)
        unit = "MB" if size > 1024**2 else "KB"
        val  = size / (1024**2 if size > 1024**2 else 1024)
        print(f"  ✅  {f:42s} {val:>8.2f} {unit}")
    else:
        print(f"  ⚠️   {f:42s} (không có)")
        missing += 1

print("\n" + "=" * 60)
if missing == 0:
    print("🎉 HOÀN TẤT — Sẵn sàng Save Version → Quick Save → Download All")
else:
    print(f"⚠️  Thiếu {missing} file — kiểm tra lại")
print("=" * 60)

KIẾN TRÚC MAXVIT (LỚP ĐẦU & LỚP CUỐI)

--- LỚP ĐẦU TIÊN (Stem / Patch Embedding) ---
Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (1): Conv2dNormActivation(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)

--- LỚP CUỐI CÙNG (Classifier Head / Filter) ---
Sequential(
  (0): AdaptiveAvgPool2d(output_size=1)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): Tanh()
  (5): Linear(in_features=512, out_features=278, bias=True)
)

BƯỚC 1: LOAD BEST MODEL
Loaded   : /kaggle/working/best_ft.pt
Saved acc: 99.43%

BƯỚC 2: INFERENCE TRÊN TẬP TEST


Eval:   0%|          | 0/776 [00:00<?, ?it/s]

Xong trong 183.4s
Avg PSNR: 21.28 dB

BƯỚC 3: BẢNG TỔNG HỢP
        Metric  Value
     Test Loss 0.0000
Top-1 Accuracy 99.43%
Top-5 Accuracy 99.74%
 Avg PSNR (dB)  21.28
   Num samples 49,640
   Num classes    278

CLASSIFICATION REPORT CHI TIẾT (278 lớp) — MAXVIT
                        precision    recall  f1-score   support

             Almonds 1     1.0000    1.0000    1.0000        77
              Apple 10     1.0000    1.0000    1.0000       231
              Apple 11     1.0000    1.0000    1.0000       142
              Apple 12     1.0000    1.0000    1.0000       154
              Apple 13     1.0000    1.0000    1.0000       235
              Apple 14     1.0000    1.0000    1.0000       154
              Apple 17     1.0000    1.0000    1.0000       201
              Apple 18     1.0000    1.0000    1.0000       240
              Apple 19     1.0000    1.0000    1.0000       241
              Apple 20     1.0000    1.0000    1.0000       234
              Apple 21     1.0

/tmp/ipykernel_23/252204768.py:328: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="Count", y="pair", data=errors_df, palette="OrRd_r", ax=ax)


Đang vẽ Đồ thị 7: Top-K Accuracy...

✅ Đã lưu Model Final → /kaggle/working/maxvit_fruit360_final.pt

CHECKLIST FILE TRONG /kaggle/working/
  ✅  maxvit_fruit360_final.pt                     117.38 MB
  ✅  summary.csv                                    0.13 KB
  ✅  maxvit_classification_report_full.txt         18.21 KB
  ✅  maxvit_classification_report.csv               9.09 KB
  ✅  maxvit_top_errors.csv                          0.42 KB
  ✅  maxvit_plot_1_training_curves.png            102.96 KB
  ✅  maxvit_plot_2_loss_vs_psnr.png                71.43 KB
  ✅  maxvit_plot_3_precision_recall_f1.png         86.45 KB
  ✅  maxvit_plot_4_f1_distribution.png             40.35 KB
  ✅  maxvit_plot_5_confusion_matrix.png           115.42 KB
  ✅  maxvit_plot_6_top_errors.png                  89.46 KB
  ✅  maxvit_plot_7_topk_accuracy.png               31.08 KB

🎉 HOÀN TẤT — Sẵn sàng Save Version → Quick Save → Download All


In [14]:
# =========================================================
# CELL 11: GOM KẾT QUẢ MAXVIT + NÉN ZIP (BẢN HOÀN CHỈNH)
# =========================================================
import os, zipfile, time
import pandas as pd

OUT_DIR = "/kaggle/working"

print("=" * 65)
print("GOM KẾT QUẢ VÀ NÉN ZIP (MAXVIT)")
print("=" * 65)

# ==========================================
# 1. DANH SÁCH FILE CẦN THIẾT
# ==========================================
files_expected = {
    "🤖 Model": [
        "maxvit_fruit360_final.pt",
        "best_head.pt",
        "best_ft.pt"
    ],
    "📊 Báo cáo": [
        "summary.csv",
        "maxvit_classification_report.csv",
        "maxvit_classification_report_full.txt",
        "maxvit_top_errors.csv",
    ],
    "📈 Đồ thị": [
        "maxvit_plot_1_training_curves.png",
        "maxvit_plot_2_loss_vs_psnr.png",
        "maxvit_plot_3_precision_recall_f1.png",
        "maxvit_plot_4_f1_distribution.png",
        "maxvit_plot_5_confusion_matrix.png",
        "maxvit_plot_6_top_errors.png",
        "maxvit_plot_7_topk_accuracy.png",
    ],
}

existing_files, missing_files = [], []
for group, files in files_expected.items():
    print(f"\n{group}:")
    for f in files:
        p = os.path.join(OUT_DIR, f)
        if os.path.exists(p):
            size = os.path.getsize(p)
            unit = "MB" if size > 1024**2 else "KB"
            val  = size / (1024**2 if size > 1024**2 else 1024)
            print(f"  ✅ {f:42s} {val:>8.2f} {unit}")
            existing_files.append(p)
        else:
            print(f"  ⚠️  {f:42s} (chưa có)")
            missing_files.append(f)

print(f"\n→ Có sẵn: {len(existing_files)} | Thiếu: {len(missing_files)}")

# ==========================================
# 2. TẠO BÁO CÁO TÓM TẮT (TXT)
# ==========================================
print("\n" + "=" * 65)
print("TẠO BÁO CÁO TÓM TẮT")
print("=" * 65)

maxvit_sum = {}
summary_path = os.path.join(OUT_DIR, "summary.csv")
if os.path.exists(summary_path):
    try:
        df = pd.read_csv(summary_path)
        maxvit_sum = dict(zip(df["Metric"], df["Value"]))
        print("✅ Đã đọc summary.csv")
    except Exception as e:
        print(f"⚠️ Lỗi đọc summary: {e}")

report_lines = [
    "=" * 70,
    "BÁO CÁO TÓM TẮT — MAXVIT PRETRAINED",
    "=" * 70,
    "",
    f"Ngày tạo: {time.strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "-" * 70,
    "1. THÔNG TIN DATASET",
    "-" * 70,
    "Dataset        : Fruit-360 100x100",
    f"Số lớp         : {len(DATA_LABELS) if 'DATA_LABELS' in globals() else 'N/A'}",
]

if 'train_N' in globals():
    report_lines.append(f"Số ảnh train   : {train_N:,}")
else:
    report_lines.append("Số ảnh train   : N/A")

if 'valid_N' in globals():
    report_lines.append(f"Số ảnh test    : {valid_N:,}")
else:
    report_lines.append("Số ảnh test    : N/A")

report_lines.extend([
    "",
    "-" * 70,
    "2. KẾT QUẢ MAXVIT",
    "-" * 70,
])

if maxvit_sum:
    for k, v in maxvit_sum.items():
        report_lines.append(f"  {k:25s}: {v}")
else:
    report_lines.append("  ⚠️ Chưa có summary")

report_lines.extend([
    "",
    "-" * 70,
    "3. FILE ĐÍNH KÈM",
    "-" * 70,
])
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    if os.path.isfile(p) and f.endswith((".png", ".csv", ".txt", ".pt")):
        size = os.path.getsize(p) / 1024
        report_lines.append(f"  {f:45s} {size:>10.1f} KB")

report_lines.extend(["", "=" * 70, "HẾT BÁO CÁO", "=" * 70])

report_text = "\n".join(report_lines)
print(report_text)

report_filename = "BAO_CAO_TOM_TAT_MAXVIT.txt"
with open(os.path.join(OUT_DIR, report_filename), "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"\n📄 Đã lưu: {report_filename}")

# ==========================================
# 3. NÉN TẤT CẢ VÀO ZIP
# ==========================================
print("\n" + "=" * 65)
print("NÉN KẾT QUẢ VÀO ZIP")
print("=" * 65)

# Gom tất cả file cần nén
all_files = []
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    if not os.path.isfile(p):
        continue
    if not f.endswith((".png", ".csv", ".txt", ".pt")):
        continue
    
    # ĐÃ SỬA: Thêm best_head.pt và best_ft.pt vào danh sách nén
    if f.startswith("maxvit_") or f in ["summary.csv", report_filename, "best_head.pt", "best_ft.pt"]:
        all_files.append(p)

zip_name = "Fruit360_MAXVIT_Results.zip"
zip_path = os.path.join(OUT_DIR, zip_name)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in all_files:
        zf.write(f, arcname=os.path.basename(f))

zip_size = os.path.getsize(zip_path) / (1024**2)
print(f"📦 Đã nén {len(all_files)} file → {zip_path}")
print(f"📊 Kích thước ZIP: {zip_size:.2f} MB")

# ==========================================
# 4. TỔNG KẾT
# ==========================================
print("\n" + "=" * 65)
print("🎯 TỔNG KẾT")
print("=" * 65)

total_expected = sum(len(files) for files in files_expected.values())
print(f"🤖 Notebook      : MAXVIT")
print(f"✅ File có sẵn   : {len(existing_files)}/{total_expected}")
print(f"⚠️  File thiếu    : {len(missing_files)}")
print(f"📦 File ZIP      : {zip_name} ({zip_size:.2f} MB)")

print("\n" + "=" * 65)
print("📌 BƯỚC TIẾP THEO:")
print("   1. Save Version → Quick Save")
print("   2. Output tab → Download All")
print(f"   3. Tải file {zip_name} về máy")
print("=" * 65)

GOM KẾT QUẢ VÀ NÉN ZIP (MAXVIT)

🤖 Model:
  ✅ maxvit_fruit360_final.pt                     117.38 MB
  ✅ best_head.pt                                 117.37 MB
  ✅ best_ft.pt                                   117.37 MB

📊 Báo cáo:
  ✅ summary.csv                                    0.13 KB
  ✅ maxvit_classification_report.csv               9.09 KB
  ✅ maxvit_classification_report_full.txt         18.21 KB
  ✅ maxvit_top_errors.csv                          0.42 KB

📈 Đồ thị:
  ✅ maxvit_plot_1_training_curves.png            102.96 KB
  ✅ maxvit_plot_2_loss_vs_psnr.png                71.43 KB
  ✅ maxvit_plot_3_precision_recall_f1.png         86.45 KB
  ✅ maxvit_plot_4_f1_distribution.png             40.35 KB
  ✅ maxvit_plot_5_confusion_matrix.png           115.42 KB
  ✅ maxvit_plot_6_top_errors.png                  89.46 KB
  ✅ maxvit_plot_7_topk_accuracy.png               31.08 KB

→ Có sẵn: 14 | Thiếu: 0

TẠO BÁO CÁO TÓM TẮT
✅ Đã đọc summary.csv
BÁO CÁO TÓM TẮT — MAXVIT PRETRAINED

Ngày 